# sweep-config-dict — worked example 1: Sweep Config — random search, maximize metric

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sweep-config-dict`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A wandb sweep config is a plain Python dict with three required top-level keys: `'method'` (search strategy), `'metric'` (what to optimize and in which direction), and `'parameters'` (one entry per hyperparameter with its distribution spec). The `'goal'` field under `'metric'` must be exactly `'minimize'` or `'maximize'`.

## Worked solution

**Step 1 — Choose the method.**
Random search samples each hyperparameter independently from its distribution. Use `'method': 'random'`.

**Step 2 — Specify the metric block.**
We want to maximize validation accuracy. The dict must have `'name'` (the metric key logged to wandb) and `'goal': 'maximize'`.

**Step 3 — Define each hyperparameter.**
For learning rate, which spans orders of magnitude, `'log_uniform_values'` samples evenly on the log scale. For number of layers, an integer count needs `'int_uniform'`. For dropout, a uniform linear range on [0, 0.5] suits the bounded probability nature of the parameter.

**Step 4 — Assemble.**
Nest all three blocks at the right keys. No wandb installation is needed — the dict shape is all that matters.

In [ ]:
def build_random_sweep_config(metric_name: str) -> dict:
    return {
        'method': 'random',
        'metric': {
            'name': metric_name,
            'goal': 'maximize',
        },
        'parameters': {
            'lr': {
                'distribution': 'log_uniform_values',
                'min': 1e-4,
                'max': 1e-1,
            },
            'num_layers': {
                'distribution': 'int_uniform',
                'min': 2,
                'max': 8,
            },
            'dropout': {
                'distribution': 'uniform',
                'min': 0.0,
                'max': 0.5,
            },
            'use_batch_norm': {
                'values': [True, False],
            },
        },
    }

# Demonstrate
cfg = build_random_sweep_config('val_accuracy')
print('method:', cfg['method'])
print('metric:', cfg['metric'])
for name, spec in cfg['parameters'].items():
    print(f'  {name}: {spec}')